# Agentic Loop

This notebook is built directly from your notes for the **Agentic Loop** lesson.

It is written in a read -> understand -> practice format, and the code cells now include **possible output comments** so you can anticipate what may happen before running them. [cite:322]


## Lesson idea

In the previous lesson, tool calls were handled manually, which made the code repetitive. This lesson introduces an **agentic loop** that keeps handling tool calls until the agent is done.

The lesson also emphasizes that every agent framework relies on some form of this loop.


In [2]:
# Install dependencies if needed.
# Run this once in a fresh environment.
%pip install -q openai gitsource minsearch pydantic

# Possible output:
# - Package installation messages
# - Or a note saying requirements are already satisfied


/workspaces/ai-engineering-buildcamp/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## Imports and setup

We import the OpenAI client, a GitHub documentation reader, a chunking helper, and a search index.


In [3]:
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex
import json
from pprint import pprint

openai_client = OpenAI()

# Possible output:
# - Usually no printed output if imports succeed
# - If credentials are missing, later API calls may fail rather than this cell


## Build the documentation index

This setup step loads the Evidently docs, parses them, chunks them, and stores them in a searchable index.


In [4]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print('Files loaded:', len(files))
print('Chunks created:', len(chunked_docs))

# Possible output:
# Files loaded: <some positive integer>
# Chunks created: <some positive integer, usually larger than files loaded>
# Exact numbers can vary if the source docs change over time


Files loaded: 95
Chunks created: 382


## Define the `search` tool

The agent needs a real Python function to run and a tool schema that tells the model how to call it.


In [5]:
def search(query):
    # Search the docs index and return the top 5 matches.
    results = index.search(query=query, num_results=5)
    return results

search_tool = {
    'type': 'function',
    'name': 'search',
    'description': 'Search the documentation database for relevant results based on a query string.',
    'parameters': {
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'The search query to look up in the index'
            }
        },
        'required': ['query']
    }
}

# Possible output:
# - No printed output
# - After this cell, the notebook has a callable search function and a search_tool schema


## The `make_call` function

This helper parses tool arguments, decides which tool to run, executes it, and returns a tool-output object in the format expected by the API.


In [6]:
def make_call(tool_call):
    # Convert the JSON argument string from the model into a Python dictionary.
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    # Decide which Python function to run based on the tool name.
    if name == 'search':
        result = search(**arguments)
    else:
        # Instead of raising an exception, return an error string.
        # This lets the agent recover if it selected the wrong tool.
        result = 'not found tool "{name}"'

    # Return the result in the exact format expected by the Responses API.
    return {
        'type': 'function_call_output',
        'call_id': tool_call.call_id,
        'output': json.dumps(result),
    }

# Possible output:
# - No printed output when defining the function
# - Later, when called with a search tool request, it returns a dictionary like:
#   {
#       'type': 'function_call_output',
#       'call_id': '...',
#       'output': '[...]'
#   }


## Manual tool handling

The lesson first revisits the older manual pattern before replacing it with a loop.


In [7]:
instructions = '''
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
'''.strip()

# Possible output:
# - No printed output
# - The variable `instructions` now contains the system prompt text


## The question contains a typo

The typo is intentional in the lesson and helps show that the model can still search effectively.


In [8]:
question = 'How do I create a dahsbord in Evidently?'

# Possible output:
# - No printed output
# - The variable `question` now stores the misspelled user query


## Create the initial message history

We start with one system message and one user message.


In [9]:
message_history = [
    {'role': 'system', 'content': instructions},
    {'role': 'user', 'content': question}
]

# Possible output:
# - No printed output unless you explicitly display `message_history`
# - The list now contains exactly 2 entries


In [10]:
message_history

[{'role': 'system',
  'content': "You're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nIMPORTANT: When you explore the knowledge base, make at least 3 different\nsearchers to make sure you explore the topic well.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'}]

## Make the first request

The model receives the message history plus the `search` tool and may return one or more tool calls.


In [11]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

# Possible output:
# - No automatic printed output unless you display `response`
# - The response will usually contain one or more items in `response.output`
# - In the lesson notes, the agent made 3 tool calls in this phase
# - Exact tool-call wording can vary between runs


In [12]:
for message in response.output:
    if message.type == "function_call":
        print(f"\nSearch: {message.arguments}")
        tool_call_output = make_call(message)
        results = json.loads(tool_call_output["output"])

        for i, result in enumerate(results, start=1):
            print(f"\nResult {i}: {result.get('title', 'Untitled')}")
            print(result.get("content", "")[:500])

        message_history.append(tool_call_output)


Search: {"query":"create dashboard"}

Result 1: Add dashboard panels (API)
You can add Panels in the user interface or using Python API. This pages describes the Python API. Check how to [add panels in the UI](dashboard_add_panels_ui).

## Dashboard Management

<Check>
  Dashboards as code are available in Evidently OSS, Cloud, Enterprise.
</Check>

<Tip>
  You must first connect to [Evidently Cloud](/docs/setup/cloud) and [create a Project](/docs/platform/projects_manage).
</Tip>

**Adding Tabs**. To add a new Tab:

```python
project.dashboard.add_tab("Another Tab")
`

Result 2: Overview
<Check>
  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.
</Check>

## What is a Dashboard?

A Dashboard provides a clear view of your AI application performance. You can use it:

- to track evaluation results across multiple experiments;
- to track live production quality over time.

Each Project has its own Dashboard. It's empty at first.

To populate 

## Extend the message history

We append the model outputs so the conversation history contains the tool calls the model just made.


In [13]:
message_history.extend(response.output)
message_history

# Possible output:
# [
#   {'role': 'system', 'content': '...'},
#   {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
#   ... model output objects ...
# ]
# The exact structure depends on the API response objects returned in this run


[{'role': 'system',
  'content': "You're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nIMPORTANT: When you explore the knowledge base, make at least 3 different\nsearchers to make sure you explore the topic well.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 {'type': 'function_call_output',
  'call_id': 'call_cC5bLoJhyGDhS2miV6f1yX5M',
  'output': '[{"start": 0, "content": "You can add Panels in the user interface or using Python API. This pages describes the Python API. Check how to [add panels in the UI](dashboard_add_panels_ui).\\n\\n## Dashboard Management\\n\\n<Check>\\n  Dashboards as code are available in Evidently OSS, Cloud, Enterprise.\\n</Check>\\n\\n<Tip>\\n  You must

## Execute the tool calls manually

This is the repetitive part: loop through each response item, run the function call, and append the tool result.


In [14]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        message_history.append(tool_call_output)

# Possible output:
# executing search({"query": "create dashboard"})...
# executing search({"query": "dashboard creation"})...
# executing search({"query": "how to create dashboard"})...
# Exact queries may vary from run to run


executing search({"query":"create dashboard"})...
executing search({"query":"dashboard creation tutorial"})...
executing search({"query":"dashboard features"})...


## Count the message history entries

Your notes say the history becomes 8 entries after the manual tool-call execution step.


In [15]:
len(message_history)

# Possible output:
# 8
# Meaning: 2 initial messages + 3 tool calls + 3 tool results


11

## Make another request to get the final answer

Now the model can read the appended tool outputs and produce an answer.


In [16]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

response.usage.input_tokens

# Possible output:
# <some integer token count>
# Example: 1200, 1800, or another value depending on the conversation length


24089

In [17]:
print(response.output_text)

# Possible output:
# A natural-language explanation answering how to create a dashboard in Evidently
# The wording can vary, but it should be grounded in the retrieved documentation


To create a dashboard in Evidently, follow these steps:

1. **Create a Project**: First, ensure you have created a project in Evidently and have connected to Evidently Cloud. You need to save at least one report to the project to populate the dashboard.

2. **Add Tabs**: By default, your dashboard will start with a single tab. You can add multiple tabs for better organization:
   - **Via Python API**:
     ```python
     project.dashboard.add_tab("Another Tab")
     ```

   - **Via UI**: 
     - Enter "Edit" mode on the dashboard.
     - Click the plus sign to add a new tab.
     - You can create a custom tab by entering a name.

3. **Add Panels**: A panel is a visual element in the dashboard that displays specific values. You can add various types of panels such as text, counters, pie charts, line plots, and bar plots. 

   - **Using the Python API**:
     ```python
     from evidently.sdk.models import PanelMetric
     from evidently.sdk.panels import DashboardPanelPlot
     
     pr

## Why the manual approach is tedious

The pattern works, but it requires the same sequence over and over again. That is the motivation for the loop.


## Update the instructions for multiple iterations

The lesson then makes the multi-step behavior more explicit by asking the model to do three iterations.


In [18]:
instructions = '''
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
'''.strip()

# Possible output:
# - No printed output
# - The new instructions encourage the model to spread work across multiple iterations


## Why use an agentic loop?

```text
Question + instructions
         |
         v
      Ask model <---------------------------+
         |                                  |
         v                                  |
  Does it request a tool?                   |
     /          \                           |
   Yes           No                          |
    |             |                          |
    v             v                          |
Run search     Show final answer             |
    |                                        |
    v                                        |
Append search results to message history ----+
```

### Why make multiple searches?

1. **First search:** Find the main dashboard-creation instructions.
2. **Follow-up searches:** Try related terms or search for details missing from the first results.
3. **Final response:** Use the collected results to answer the question.

The loop lets the model inspect search results and decide whether it needs more information. The prompt asks for three stages, but the loop itself continues until the model stops requesting tools; it isn’t hard-coded to exactly three iterations.

## Now define the agentic loop

This loop is the key idea of the lesson: keep going until the model no longer requests tools.


In [19]:
message_history = [
    {'role': 'system', 'content': instructions},
    {'role': 'user', 'content': question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...')
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

# Possible output:
# iteraration number 1...
# executing search({"query": "create dashboard"})...
#
# iteraration number 2...
# ASSISTANT: I first want to search for the core phrase...
# executing search({"query": "dashboard panels"})...
# executing search({"query": "dashboard report setup"})...
#
# iteraration number 3...
# ASSISTANT: Based on the documentation...
#
# Exact wording and search queries can vary


iteraration number 1...
ASSISTANT: To provide you with the most accurate information on creating a dashboard, I will first perform a search for "creating a dashboard." This will help me identify the specific steps or guidelines available in the documentation related to dashboard creation.
executing search({"query":"creating a dashboard"})...

iteraration number 2...
ASSISTANT: The first search revealed information about adding panels and tabs to a dashboard through both a user interface and a Python API. Here’s a quick overview of the key points:

1. **Dashboards**: You can create dashboards that visualize evaluation results. It's essential to add reports with results to the project first.
2. **Adding Tabs**: You can organize your dashboard by adding tabs. This can be done in "Edit" mode.
3. **Adding Panels**: Various types of panels (text, counters, plots) can be added to visualize data.

Next, I'll perform two additional searches. The first search will focus on more general dashboard

## What the loop is doing

The loop repeatedly sends the full history, executes function calls, appends results, prints assistant messages, and stops only when no more function calls are requested.


## Add another tool

Once the loop exists, the lesson shows that it becomes easier to add more tools.


In [ ]:
def add_entry(filename, title, description, content):
    # Build a new index entry with the structure expected by the search index.
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }

    # Append the new entry into the existing documentation index.
    index.append(entry)
    return 'OK'

add_entry_tool = {
    'type': 'function',
    'name': 'add_entry',
    'description': 'Add a new documentation entry to the index.',
    'parameters': {
        'type': 'object',
        'properties': {
            'filename': {
                'type': 'string',
                'description': 'The source filename associated with the entry'
            },
            'title': {
                'type': 'string',
                'description': 'The title of the documentation entry'
            },
            'description': {
                'type': 'string',
                'description': 'A short description summarizing the entry'
            },
            'content': {
                'type': 'string',
                'description': 'The full content of the documentation entry'
            }
        },
        'required': ['filename', 'title', 'description', 'content']
    }
}

# Possible output:
# - No printed output
# - After this cell, the notebook has a second tool named add_entry


## Update `make_call` to support the new tool

The routing function now supports both tool names.


In [ ]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else:
        result = 'not found tool "{name}"'

    return {
        'type': 'function_call_output',
        'call_id': tool_call.call_id,
        'output': json.dumps(result),
    }

# Possible output:
# - No printed output during definition
# - When called later, it will return a tool-output dictionary for either search or add_entry


## Multiple tools working together

We now run the same loop pattern, but with both tools available.


In [ ]:
message_history = [
    {'role': 'system', 'content': instructions},
    {'role': 'user', 'content': question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...')
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

# Possible output:
# iteraration number 1...
# executing search({...})...
# ...
# ASSISTANT: <research summary or reasoning text>
#
# In this phase, the model may still mostly use search before it uses add_entry


## Ask the agent to save the content

The next user message explicitly tells the agent to add the content to the database.


In [ ]:
message_history.append(
    {'role': 'user', 'content': 'add this content to our database'}
)

# Possible output:
# - No printed output
# - The conversation history now includes a follow-up request asking the agent to persist the content


## Run the loop again

Now the model has a reason to use the `add_entry` tool.


In [ ]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...')
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

# Possible output:
# iteraration number <n>...
# executing add_entry({"filename": ..., "title": ..., "description": ..., "content": ...})...
# ASSISTANT: The content has been added to the database.
# Exact argument values and response wording may vary


## Verify the last entry

The lesson ends by checking the final indexed document.


In [ ]:
index.docs[-1]

# Possible output:
# {
#   'start': 0,
#   'content': '...',
#   'title': '...',
#   'description': '...',
#   'filename': '...'
# }
# This confirms that a new document entry was appended to the index


## Main takeaway

The key lesson is the reusable pattern: call the model, execute requested tools, append outputs, and keep looping until the model stops asking for tools.


## Practice tasks

Try these after reading and running the notebook:
1. Add a max-iteration guard
2. Log each tool call to a Python list
3. Add a third tool such as `get_doc_by_filename`
4. Refactor the loop into a helper like `run_agent_loop(...)`
5. Return structured JSON at the end
